In [1]:
"""Prueba de acceso a ClickHouse: conexión, bases de datos y una tabla. Solo lectura y solo
recuentos y esquema, nunca filas de jugadores."""

import os
from getpass import getpass

import clickhouse_connect
from dotenv import load_dotenv

load_dotenv()  # lee .env si existe


client = clickhouse_connect.get_client(
    host=os.environ["CLICKHOUSE_HOST"],
    port=int(os.getenv("CLICKHOUSE_PORT", "28123")),
    username=os.environ["CLICKHOUSE_USER"],
    password=os.getenv("CLICKHOUSE_PASSWORD") or getpass("ClickHouse password: "),
    secure=False,
)

print(client.command("SELECT version()"))
print(client.command("SELECT currentUser()"))
print(client.command("SELECT getSetting('readonly')"))   # 1 o 2: tu usuario es de solo lectura


print("\nDatabases:")
for (db,) in client.query("SHOW DATABASES").result_rows:
    print(" -", db)

# tablas de gold relacionadas con las señales
tables = client.query(
    "SELECT database, name, total_rows FROM system.tables "
    "WHERE name LIKE 'gld_player%' ORDER BY database, name"
).result_rows
print("\nGold player tables:")
for db, name, rows in tables:
    print(f" - {db}.{name}: {rows:,} rows" if rows is not None else f" - {db}.{name}")

# una tabla concreta: esquema y recuento por día (agregados, sin filas de jugadores)
if tables:
    db, name, _ = next((t for t in tables if t[1] == "gld_player_signals_daily"), tables[0])
    print(f"\nColumns of {db}.{name}:")
    for col, typ in client.query(f"SELECT name, type FROM system.columns "
                                 f"WHERE database = '{db}' AND table = '{name}' LIMIT 10").result_rows:
        print(f" - {col}: {typ}")
    print(client.query_df(f"SELECT toDate(snapshot_date) AS day, count() AS rows FROM {db}.{name} "
                          f"GROUP BY day ORDER BY day DESC LIMIT 5")
          if name == "gld_player_signals_daily" else
          client.command(f"SELECT count() FROM {db}.{name}"))


26.4.2.10
data_scientist_readonly
2

Databases:
 - whizdomai_org_gmntc_gld
 - whizdomai_org_gmntc_medallion

Gold player tables:
 - whizdomai_org_gmntc_medallion.gld_player_bonus_daily: 206,003,329 rows
 - whizdomai_org_gmntc_medallion.gld_player_bonuses: 23,030,804 rows
 - whizdomai_org_gmntc_medallion.gld_player_engagement_daily: 787,960,536 rows
 - whizdomai_org_gmntc_medallion.gld_player_financial_daily: 816,590,048 rows
 - whizdomai_org_gmntc_medallion.gld_player_gaming_daily: 68,599,689 rows
 - whizdomai_org_gmntc_medallion.gld_player_lifetime: 2,524,552 rows
 - whizdomai_org_gmntc_medallion.gld_player_lifetime_current
 - whizdomai_org_gmntc_medallion.gld_player_lifetime_current_da09_scratch
 - whizdomai_org_gmntc_medallion.gld_player_lifetime_da09_scratch: 682,186 rows
 - whizdomai_org_gmntc_medallion.gld_player_live: 682,408 rows
 - whizdomai_org_gmntc_medallion.gld_player_payments: 20,876,500 rows
 - whizdomai_org_gmntc_medallion.gld_player_payments_daily: 6,587,555 rows
 - wh